# XGBoost를 활용한 암종 분류

암환자의 4,384개 유전자 변이 정보를 이용해 26개 `SUBCLASS`를 분류하는 XGBoost baseline Notebook입니다.

기존 `[Baseline]_XGB를 활용한 암종 분류 AI 모델 개발.ipynb`의 **대회 소개 → 라이브러리 → 데이터 로드 → 전처리 → 모델 학습 → 추론 → 제출** 흐름을 모두 통합하고, 현재 저장소의 데이터 계약·공용 split·재현성 규칙에 맞게 보강했습니다.

> 이 파일 자체는 아직 실행된 모델 실험이 아닙니다. 기본값 `RUN_MODE = "explore"`는 데이터와 전처리까지만 검증합니다. 실제 학습은 GitHub Experiment Issue 브랜치에서 `RUN_MODE = "experiment"`로 바꾸면 Issue 번호 기반 EXP-ID가 자동 생성됩니다.

## 0. 대회 목적과 Notebook 개선점

### 기존 baseline의 대회 설명

- 바이오 데이터를 기반으로 한 AI 기술의 문제 해결 능력을 탐구하는 것을 목표로 합니다. 이 대회는 바이오 분야에서 AI 활용의 저변을 확대하고, 복잡한 바이오 데이터를 효율적으로 분석 및 해석할 수 있는 AI 알고리즘 개발에 초점을 맞추고 있습니다.
- 본 대회의 구체적인 과제는 암환자 유전체 데이터의 변이 정보를 활용하여 암종을 분류하는 AI 모델을 개발하는 것입니다. 참가자들은 제공된 학습 데이터셋을 사용하여 특정 변이 정보를 바탕으로 암종을 정확하게 분류할 수 있는 AI 알고리즘을 개발해야 합니다.
- 이 대회의 궁극적인 목적은 바이오 데이터의 활용도를 높이고, 바이오 분야에서 AI 기술의 적용 가능성을 극대화하며, 인공지능 기술이 실제 바이오 의료 문제 해결에 어떻게 기여할 수 있는지 탐구하는 것입니다.

### 통합하면서 개선한 내용

원본 baseline의 개념은 유지하면서 다음을 개선했습니다.

- 현재 실행 위치와 관계없이 저장소 루트와 `data/raw/`를 자동 탐색합니다.
- 학습 전에 행·열 수, 파일 SHA-256, ID, 클래스와 유전자 순서를 검증합니다.
- 임의 hold-out 대신 팀 공용 `stratified_5fold_seed42.csv`를 사용합니다.
- 공식 지표인 전체 OOF Macro F1과 fold별 Macro F1을 계산합니다.
- 원본의 `LabelEncoder`와 `OrdinalEncoder` 방식을 사용할 수 있게 유지합니다.
- 기본 인코딩은 데이터의 희소성을 살리는 비-WT 변이 존재 여부 CSR 행렬입니다.
- train에서만 전처리를 fit하고 validation/test는 transform만 하도록 데이터 누출을 방지합니다.
- OOF, 테스트 확률, checkpoint와 제출 파일의 이름과 경로를 분리합니다.
- 제출 전 ID 순서와 허용 클래스 검증을 자동 수행합니다.

## 1. 실행 전 준비

저장소 루트에서 `uv sync --frozen`을 한 번 실행하고 VS Code/Jupyter kernel로 `.venv`의 Python을 선택합니다. 원본 CSV는 저장소의 `data/raw/`에 포함되어 있습니다.

실제 학습을 시작할 때는 다음 순서를 지킵니다.

1. GitHub 실험 Issue를 생성합니다.
2. 발급된 번호로 `12` 또는 `issue-12-exp-xgb-baseline` 같은 브랜치를 만듭니다.
3. 아래 설정에서 `RUN_MODE = "experiment"`로 바꿉니다. Issue #12라면 `EXP-012`가 자동 생성됩니다.
4. Notebook에서 확인한 로직을 `configs/`와 `scripts/`로 옮긴 뒤 공식 실험을 실행합니다.
5. 실제 결과와 재현성 증빙을 History에 기록합니다.

In [ ]:
from __future__ import annotations

import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from scipy import sparse
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.utils.class_weight import compute_sample_weight

from open_cancer.constants import CLASS_LABELS, PROBABILITY_COLUMNS
from open_cancer.experiment import resolve_experiment_context
from open_cancer.validation import validate_competition_data, validate_submission

SEED = 42
N_SPLITS = 5
RUN_MODE = "explore"  # "explore" 또는 "experiment"
ENCODING_METHOD = "mutation_presence"  # "mutation_presence" 또는 "ordinal"
USE_BALANCED_SAMPLE_WEIGHT = True
N_JOBS = max(1, min(8, os.cpu_count() or 1))

experiment_context = resolve_experiment_context(RUN_MODE, cwd=Path.cwd())
ISSUE_NUMBER = experiment_context.issue_number
EXPERIMENT_ID = experiment_context.experiment_id
RUN_TRAINING = experiment_context.is_experiment

if ENCODING_METHOD not in {"mutation_presence", "ordinal"}:
    raise ValueError("ENCODING_METHOD는 mutation_presence 또는 ordinal이어야 합니다.")

random.seed(SEED)
np.random.seed(SEED)

{
    "python_seed": SEED,
    "xgboost_version": xgb.__version__,
    "run_mode": RUN_MODE,
    "branch": experiment_context.branch,
    "issue_number": ISSUE_NUMBER,
    "experiment_id": EXPERIMENT_ID,
    "encoding": ENCODING_METHOD,
    "training_enabled": RUN_TRAINING,
    "n_jobs": N_JOBS,
}

In [ ]:
def find_project_root(start: Path) -> Path:
    """pyproject.toml과 PROJECT_CONTEXT.md가 있는 저장소 루트를 찾습니다."""
    resolved = start.resolve()
    for candidate in (resolved, *resolved.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "PROJECT_CONTEXT.md"
        ).is_file():
            return candidate
    raise FileNotFoundError("open_cancer 저장소 안에서 Notebook을 실행하세요.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"
FOLD_PATH = PROJECT_ROOT / "data" / "splits" / "stratified_5fold_seed42.csv"

{
    "project_root": str(PROJECT_ROOT),
    "train": str(TRAIN_PATH.relative_to(PROJECT_ROOT)),
    "test": str(TEST_PATH.relative_to(PROJECT_ROOT)),
    "fold": str(FOLD_PATH.relative_to(PROJECT_ROOT)),
}

## 2. 원본 데이터 검증과 로드

파일을 읽기 전에 프로젝트의 고정 데이터 계약을 검증합니다. 그다음 모든 변이 값을 문자열로 유지하고, 원본 train 순서를 보존하면서 공용 fold를 `ID`로 연결합니다.

In [ ]:
data_summary = validate_competition_data(
    TRAIN_PATH,
    TEST_PATH,
    SAMPLE_SUBMISSION_PATH,
)

train_raw = pd.read_csv(TRAIN_PATH, dtype=str, keep_default_na=False)
test = pd.read_csv(TEST_PATH, dtype=str, keep_default_na=False)
sample_submission = pd.read_csv(
    SAMPLE_SUBMISSION_PATH,
    dtype=str,
    keep_default_na=False,
)
folds = pd.read_csv(FOLD_PATH, dtype={"ID": str, "fold": int})

train = train_raw.merge(folds, on="ID", how="left", validate="one_to_one", sort=False)
if not train["ID"].equals(train_raw["ID"]):
    raise ValueError("fold 병합 과정에서 train 행 순서가 바뀌었습니다.")
if train["fold"].isna().any() or set(train["fold"]) != set(range(N_SPLITS)):
    raise ValueError("공용 fold가 모든 train ID에 0~4로 배정되지 않았습니다.")
if not sample_submission["ID"].equals(test["ID"]):
    raise ValueError("sample_submission과 test의 ID 값 또는 순서가 다릅니다.")

gene_columns = [column for column in test.columns if column != "ID"]
train_gene_columns = [
    column for column in train.columns if column not in {"ID", "SUBCLASS", "fold"}
]
if gene_columns != train_gene_columns:
    raise ValueError("train/test 유전자 컬럼 이름 또는 순서가 다릅니다.")

data_summary

In [ ]:
data_overview = {
    "train_shape_with_fold": train.shape,
    "test_shape": test.shape,
    "gene_columns": len(gene_columns),
    "fold_counts": train["fold"].value_counts().sort_index().to_dict(),
    "class_counts": train["SUBCLASS"].value_counts().sort_index().to_dict(),
    "train_blank_cells": int((train[gene_columns] == "").to_numpy().sum()),
    "test_blank_cells": int((test[gene_columns] == "").to_numpy().sum()),
}
data_overview

## 3. 타깃 인코딩

원본 baseline처럼 `LabelEncoder`를 사용하되, 학습 데이터에서 우연히 얻은 순서에 의존하지 않도록 프로젝트의 고정 26개 클래스에 fit합니다. 예측 확률의 열 순서는 항상 `CLASS_LABELS`와 동일합니다.

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(list(CLASS_LABELS))
if list(label_encoder.classes_) != list(CLASS_LABELS):
    raise ValueError("LabelEncoder 클래스 순서가 프로젝트 고정 순서와 다릅니다.")

y = label_encoder.transform(train["SUBCLASS"]).astype(np.int32)
label_mapping = pd.DataFrame(
    {
        "SUBCLASS": label_encoder.classes_,
        "encoded": np.arange(len(label_encoder.classes_)),
    }
)
display(label_mapping)

## 4. 유전자 변이 피처 인코딩

원본 baseline은 각 유전자 문자열에 `OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)`를 적용했습니다. 이 방식은 아래 `ordinal` 옵션으로 보존하되, fold마다 **fold train에만 fit**하여 validation 정보가 들어가지 않게 합니다.

기본값 `mutation_presence`는 각 셀을 다음처럼 인코딩합니다.

- `WT` 또는 빈 문자열: `0`
- 그 밖의 변이 문자열: `1`

이 데이터는 대부분 `WT`이므로 CSR 희소 행렬을 사용하면 메모리와 학습 시간을 크게 줄일 수 있습니다. 한 셀 안의 세부 변이 문자열을 구분하지 않는 단순 baseline이라는 한계는 후속 실험에서 개선합니다.

In [ ]:
def mutation_presence_matrix(frame: pd.DataFrame) -> sparse.csr_matrix:
    """WT/빈 값은 0, 변이 문자열은 1인 float32 CSR 행렬을 만듭니다."""
    values = frame.loc[:, gene_columns]
    mutation_mask = values.ne("WT") & values.ne("")
    return sparse.csr_matrix(
        mutation_mask.to_numpy(dtype=np.float32, copy=False),
        dtype=np.float32,
    )


def ordinal_fold_matrices(
    fold_train: pd.DataFrame,
    fold_valid: pd.DataFrame,
    test_frame: pd.DataFrame,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, OrdinalEncoder]:
    """원본 baseline의 OrdinalEncoder를 fold 누출 없이 적용합니다."""
    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        dtype=np.float32,
    )
    x_train = encoder.fit_transform(fold_train.loc[:, gene_columns])
    x_valid = encoder.transform(fold_valid.loc[:, gene_columns])
    x_test = encoder.transform(test_frame.loc[:, gene_columns])
    return x_train, x_valid, x_test, encoder

In [ ]:
if ENCODING_METHOD == "mutation_presence":
    x_all = mutation_presence_matrix(train)
    x_test_shared = mutation_presence_matrix(test)
    feature_summary = {
        "train_matrix_shape": x_all.shape,
        "test_matrix_shape": x_test_shared.shape,
        "train_nonzero": int(x_all.nnz),
        "test_nonzero": int(x_test_shared.nnz),
        "train_density": float(x_all.nnz / np.prod(x_all.shape)),
        "test_density": float(x_test_shared.nnz / np.prod(x_test_shared.shape)),
    }
else:
    x_all = None
    x_test_shared = None
    feature_summary = {
        "encoding": "ordinal",
        "note": "각 fold 학습 시 fold train에 encoder를 fit합니다.",
    }

feature_summary

## 5. XGBoost 모델과 공용 5-fold 학습

원본 baseline의 `n_estimators=100`, `learning_rate=0.1`, `max_depth=6`, `random_state=42`, `eval_metric='mlogloss'` 설정을 출발점으로 삼았습니다. 여기에 multi-class objective, CPU `hist`, subsampling, 정규화와 early stopping을 명시합니다.

Macro F1은 미분 가능한 학습 objective가 아니므로 학습과 early stopping에는 multi-class log loss를 사용하고, 모델 선택 결과는 OOF Macro F1로 비교합니다.

In [ ]:
XGB_PARAMS = {
    "objective": "multi:softprob",
    "num_class": len(CLASS_LABELS),
    "n_estimators": 500,
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 1.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "eval_metric": "mlogloss",
    "early_stopping_rounds": 30,
    "tree_method": "hist",
    "device": "cpu",
    "n_jobs": N_JOBS,
    "verbosity": 0,
}
XGB_PARAMS

In [ ]:
fold_results: list[dict[str, int | float | None]] = []
trained_models: list[xgb.XGBClassifier] = []
oof_proba = np.full((len(train), len(CLASS_LABELS)), np.nan, dtype=np.float32)
test_proba = np.zeros((len(test), len(CLASS_LABELS)), dtype=np.float32)

if RUN_TRAINING:
    artifact_slug = f"{EXPERIMENT_ID.lower().replace('-', '')}_xgb_baseline"
    model_dir = PROJECT_ROOT / "models" / artifact_slug
    model_dir.mkdir(parents=True, exist_ok=True)

    for fold in range(N_SPLITS):
        valid_mask = train["fold"].eq(fold).to_numpy()
        train_indices = np.flatnonzero(~valid_mask)
        valid_indices = np.flatnonzero(valid_mask)

        if ENCODING_METHOD == "mutation_presence":
            x_train_fold = x_all[train_indices]
            x_valid_fold = x_all[valid_indices]
            x_test_fold = x_test_shared
        else:
            (
                x_train_fold,
                x_valid_fold,
                x_test_fold,
                _ordinal_encoder,
            ) = ordinal_fold_matrices(
                train.iloc[train_indices],
                train.iloc[valid_indices],
                test,
            )

        y_train_fold = y[train_indices]
        y_valid_fold = y[valid_indices]
        sample_weight = (
            compute_sample_weight(class_weight="balanced", y=y_train_fold)
            if USE_BALANCED_SAMPLE_WEIGHT
            else None
        )

        model = xgb.XGBClassifier(
            **XGB_PARAMS,
            random_state=SEED + fold,
        )
        model.fit(
            x_train_fold,
            y_train_fold,
            sample_weight=sample_weight,
            eval_set=[(x_valid_fold, y_valid_fold)],
            verbose=False,
        )

        if not np.array_equal(model.classes_, np.arange(len(CLASS_LABELS))):
            raise ValueError(f"fold {fold} 모델의 확률 클래스 순서가 다릅니다.")

        valid_proba = model.predict_proba(x_valid_fold).astype(np.float32)
        fold_test_proba = model.predict_proba(x_test_fold).astype(np.float32)
        oof_proba[valid_indices] = valid_proba
        test_proba += fold_test_proba / N_SPLITS

        valid_predictions = valid_proba.argmax(axis=1)
        fold_macro_f1 = f1_score(y_valid_fold, valid_predictions, average="macro")
        best_iteration = getattr(model, "best_iteration", None)
        fold_results.append(
            {
                "fold": fold,
                "macro_f1": float(fold_macro_f1),
                "best_iteration": None if best_iteration is None else int(best_iteration),
                "train_rows": len(train_indices),
                "valid_rows": len(valid_indices),
            }
        )
        model.save_model(model_dir / f"fold_{fold:02d}.json")
        trained_models.append(model)
        print(
            f"fold={fold} macro_f1={fold_macro_f1:.6f} "
            f"best_iteration={best_iteration}"
        )

    if np.isnan(oof_proba).any():
        raise ValueError("OOF 확률에 채워지지 않은 행이 있습니다.")
else:
    print(
        "학습을 건너뜁니다. Experiment Issue 번호가 있는 브랜치에서 "
        "RUN_MODE='experiment'로 바꾸면 EXP-ID가 자동 생성됩니다."
    )

## 6. OOF Macro F1 평가

5개 validation fold를 모두 합친 OOF 예측으로 공식 지표와 같은 Macro F1을 계산합니다. fold 평균뿐 아니라 전체 OOF 점수와 클래스별 F1을 함께 확인합니다. 아래 출력은 실제 학습을 실행했을 때만 생성됩니다.

In [ ]:
if RUN_TRAINING:
    oof_predictions = oof_proba.argmax(axis=1)
    oof_macro_f1 = f1_score(y, oof_predictions, average="macro")
    class_report = classification_report(
        y,
        oof_predictions,
        labels=np.arange(len(CLASS_LABELS)),
        target_names=CLASS_LABELS,
        output_dict=True,
        zero_division=0,
    )
    class_f1 = pd.DataFrame(
        {
            "SUBCLASS": CLASS_LABELS,
            "f1": [class_report[label]["f1-score"] for label in CLASS_LABELS],
            "support": [int(class_report[label]["support"]) for label in CLASS_LABELS],
        }
    )
    fold_metrics = pd.DataFrame(fold_results)
    print(f"전체 OOF Macro F1: {oof_macro_f1:.6f}")
    display(fold_metrics)
    display(class_f1)
else:
    print("RUN_MODE='explore'이므로 측정된 모델 점수는 없습니다.")

## 7. 테스트 추론과 제출 파일 생성

각 fold 모델의 test 확률을 동일 가중치로 평균하고, 가장 높은 확률의 클래스를 원래 `SUBCLASS` 문자열로 복원합니다. 원본 baseline의 `submisson` 오타와 `X_encoded` 변수 덮어쓰기를 바로잡았으며, 제출 저장 직후 프로젝트 검증 함수를 실행합니다.

In [ ]:
if RUN_TRAINING:
    artifact_slug = f"{EXPERIMENT_ID.lower().replace('-', '')}_xgb_baseline"
    oof_dir = PROJECT_ROOT / "oof"
    preds_dir = PROJECT_ROOT / "preds"
    submissions_dir = PROJECT_ROOT / "submissions"
    report_dir = PROJECT_ROOT / "reports" / artifact_slug
    for directory in (oof_dir, preds_dir, submissions_dir, report_dir):
        directory.mkdir(parents=True, exist_ok=True)

    oof_predictions = oof_proba.argmax(axis=1)
    oof_frame = pd.DataFrame(
        {
            "ID": train["ID"],
            "SUBCLASS_TRUE": train["SUBCLASS"],
            "SUBCLASS_PRED": label_encoder.inverse_transform(oof_predictions),
            "FOLD": train["fold"].astype(int),
        }
    )
    oof_frame.loc[:, list(PROBABILITY_COLUMNS)] = oof_proba

    test_probability_frame = pd.DataFrame({"ID": test["ID"]})
    test_probability_frame.loc[:, list(PROBABILITY_COLUMNS)] = test_proba

    submission = sample_submission.copy()
    submission["SUBCLASS"] = label_encoder.inverse_transform(test_proba.argmax(axis=1))

    oof_path = oof_dir / f"{artifact_slug}.csv"
    test_probability_path = preds_dir / f"{artifact_slug}_test_proba.csv"
    submission_path = submissions_dir / f"{artifact_slug}.csv"
    oof_frame.to_csv(oof_path, index=False, lineterminator="\n")
    test_probability_frame.to_csv(
        test_probability_path,
        index=False,
        lineterminator="\n",
    )
    submission.to_csv(submission_path, index=False, lineterminator="\n")

    submission_validation = validate_submission(submission_path, TEST_PATH)
    notebook_run_summary = {
        "issue_number": ISSUE_NUMBER,
        "experiment_id": EXPERIMENT_ID,
        "branch": experiment_context.branch,
        "seed": SEED,
        "encoding_method": ENCODING_METHOD,
        "balanced_sample_weight": USE_BALANCED_SAMPLE_WEIGHT,
        "xgboost_version": xgb.__version__,
        "xgb_params": XGB_PARAMS,
        "fold_results": fold_results,
        "oof_macro_f1": float(oof_macro_f1),
        "data_files": data_summary["files"],
        "feature_order_sha256": data_summary["feature_order_sha256"],
        "submission_validation": submission_validation,
    }
    summary_path = report_dir / "notebook_run.json"
    summary_path.write_text(
        json.dumps(notebook_run_summary, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(f"OOF: {oof_path.relative_to(PROJECT_ROOT)}")
    print(f"Test probabilities: {test_probability_path.relative_to(PROJECT_ROOT)}")
    print(f"Submission: {submission_path.relative_to(PROJECT_ROOT)}")
    print(f"Summary: {summary_path.relative_to(PROJECT_ROOT)}")
    display(submission.head())
    submission_validation
else:
    print("학습하지 않았으므로 OOF, test 확률, checkpoint와 제출 파일을 만들지 않았습니다.")

## 8. 공식 실험으로 전환할 때

이 Notebook에는 원본 baseline의 모든 기능이 통합되어 있지만, 리더보드에 사용할 실험은 Notebook 출력만으로 끝내지 않습니다.

1. 기본값과 필요한 override를 합친 resolved config를 저장합니다.
2. 전처리·학습·추론을 `scripts/run_expNNN_<slug>.py`로 옮깁니다.
3. OOF Macro F1과 실제 산출물을 `EXPERIMENT_HISTORY.md`에 기록합니다.
4. 리더보드에 제출할 때만 환경·데이터·산출물 manifest를 완성합니다.
5. 제출 모델은 checkpoint 추론으로 제출 파일을 동일하게 재생성합니다.

후속 개선 후보는 변이 문자열 token/count 피처, 유전자별 빈도 필터링, class weight 비교, XGBoost 튜닝과 다른 모델의 확률 앙상블입니다.